# Modelo operativo PISCO — Huánuco (Google Colab)

Entrena el modelo de lluvia que usa **solo PISCOp 1981→presente** (+ ONI) con evaluación *walk-forward* 2016–2025 y emite el pronóstico de los próximos 3 meses con probabilidades por terciles.

1. Ejecuta la celda 1 y sube `paquete_pisco_colab.zip` (carpeta `colab/` del proyecto).
2. *Entorno de ejecución → Ejecutar todas*. Tarda ~1 minuto; no necesita GPU.
3. Al final se descarga `resultados_pisco.zip`: descomprímelo en la carpeta del proyecto.

**Importante:** los scripts se ejecutan con `!python script.py`, no pegando su código en una celda.

## 1. Subir el paquete

In [ ]:
from google.colab import files
import zipfile
subido = files.upload()                      # elige paquete_pisco_colab.zip
zipfile.ZipFile(next(iter(subido))).extractall('.')
!ls -la

## 2. Dependencias (Colab ya las trae; solo se confirman versiones)

In [ ]:
!pip -q install xgboost scikit-learn xarray scipy joblib
import pandas, sklearn, xgboost, xarray
print('pandas', pandas.__version__, '| sklearn', sklearn.__version__, '| xgboost', xgboost.__version__, '| xarray', xarray.__version__)

## 3. Entrenar, evaluar (walk-forward) y pronosticar

In [ ]:
!python 07_modelo_pisco.py

## 4. Revisar resultados

In [ ]:
import pandas as pd, json
res = pd.read_csv('resultados_pisco/metricas_resumen.csv')
display(res.pivot_table(index='modelo', columns='h', values=['SS_clim', 'RMSE', 'ACC']).round(3))
display(pd.read_csv('resultados_pisco/rpss.csv').round(3))
pron = pd.read_csv('resultados_pisco/pronostico_actual.csv')
display(pron.groupby(['fecha_obj', 'categoria']).size().unstack(fill_value=0))
print(json.load(open('resultados_pisco/config.json'))['modelo_por_horizonte'])

## 5. Descargar resultados

In [ ]:
!zip -qr resultados_pisco.zip resultados_pisco
files.download('resultados_pisco.zip')

## 6. (Opcional) Variantes
`--ini 1991-01` (serie más corta), `--anios 2019-2025`, `--sin-oni`, `--salida otra_carpeta`.

In [ ]:
# !python 07_modelo_pisco.py --ini 2001-01 --salida resultados_pisco_2001